# Ulasan pengguna untuk analisis sentimen pada game Genshin Impact

In [34]:
import pandas as pd
import re 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import RandomOverSampler

slang_df = pd.read_csv('./datasets/slang_dict.csv')
slang_dict = dict(zip(slang_df['slang'], slang_df['formal']))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text) 
    words = text.split()
    normalized_words = [slang_dict.get(word, word) for word in words]
    return ' '.join(normalized_words)

def labeling(score):
    if score <= 2: return 0
    elif score == 3: return 1
    else: return 2

df = pd.read_csv("./datasets/scrapping_result.csv")
df['cleaned_content'] = df['content'].apply(clean_text)
df['label'] = df['score'].apply(labeling)

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(df[['cleaned_content']], df['label'])

def run_optimized_setup(name, model, vectorizer, split_ratio):
    X_train, X_test, y_train, y_test = train_test_split(
        X_resampled['cleaned_content'], y_resampled, test_size=1-split_ratio, random_state=42
    )
    
    pipeline = Pipeline([
        ('tfidf', vectorizer),
        ('clf', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    print(f"=== {name} ===")
    print(f"Accuracy Train: {pipeline.score(X_train, y_train):.4f}")
    print(f"Accuracy Test: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))
    return pipeline

tfidf_bigram = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_trigram = TfidfVectorizer(max_features=10000, ngram_range=(1, 3))

def predict_sentiment(text, model_pipeline):
    cleaned = clean_text(text)
    prediction = model_pipeline.predict([cleaned])[0]
    mapping = {0: "Negatif", 1: "Netral", 2: "Positif"}
    return mapping[prediction]


In [35]:
svm_model_bigram = run_optimized_setup("SVM + Bigram + Oversampling", LinearSVC(), tfidf_bigram, 0.8)

test_review = "Gamenya seru bgt tapi gachanya ampas bikin emosi"
print(f"\nReview: {test_review}")
print(f"Hasil Sentimen: {predict_sentiment(test_review, svm_model_bigram)}")

=== SVM + Bigram + Oversampling ===
Accuracy Train: 0.9235
Accuracy Test: 0.8671
              precision    recall  f1-score   support

           0       0.82      0.87      0.84      1525
           1       0.90      0.94      0.92      1595
           2       0.88      0.79      0.84      1584

    accuracy                           0.87      4704
   macro avg       0.87      0.87      0.87      4704
weighted avg       0.87      0.87      0.87      4704


Review: Gamenya seru bgt tapi gachanya ampas bikin emosi
Hasil Sentimen: Positif


In [36]:
svm_model_trigram = run_optimized_setup("SVM + Trigram + Oversampling", LinearSVC(C=1.0), tfidf_trigram, 0.7)

test_review = "Gamenya seru bgt tapi gachanya ampas bikin emosi"
print(f"\nReview: {test_review}")
print(f"Hasil Sentimen: {predict_sentiment(test_review, svm_model_trigram)}")

=== SVM + Trigram + Oversampling ===
Accuracy Train: 0.9400
Accuracy Test: 0.8774
              precision    recall  f1-score   support

           0       0.83      0.88      0.86      2328
           1       0.92      0.95      0.93      2398
           2       0.88      0.80      0.84      2331

    accuracy                           0.88      7057
   macro avg       0.88      0.88      0.88      7057
weighted avg       0.88      0.88      0.88      7057


Review: Gamenya seru bgt tapi gachanya ampas bikin emosi
Hasil Sentimen: Positif


In [37]:
random_forest_model = run_optimized_setup("Random Forest + Unigram + Oversampling", RandomForestClassifier(n_estimators=100), tfidf_unigram, 0.8)

test_review = "Gamenya seru bgt tapi gachanya ampas bikin emosi"
print(f"\nReview: {test_review}")
print(f"Hasil Sentimen: {predict_sentiment(test_review, random_forest_model)}")

=== Random Forest + Unigram + Oversampling ===
Accuracy Train: 0.9744
Accuracy Test: 0.9101
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      1525
           1       0.99      0.96      0.97      1595
           2       0.91      0.85      0.88      1584

    accuracy                           0.91      4704
   macro avg       0.91      0.91      0.91      4704
weighted avg       0.91      0.91      0.91      4704


Review: Gamenya seru bgt tapi gachanya ampas bikin emosi
Hasil Sentimen: Positif
